## Optional upgraded frontend — Streamlit dashboard

A separate polished browser interface is included as `arsenal_dashboard.py`. Install packages from `requirements_arsenal.txt` in Anaconda Prompt and run `streamlit run arsenal_dashboard.py`. The notebook remains available for experimentation. Older API seasons returning 403 require a different licensed/public historical source; changing the year slider cannot bypass access restrictions.

# Arsenal Match Predictor 2.2 — Select your competition

Choose **one** league/cup, then download **only that competition**. Historical access depends on your football-data.org subscription. Existing model features, evaluations and predictions are preserved.

**Run cells in order.** In the selector cell, choose a competition and season range, then run the download cell. A fresh selection replaces the active training data, not your disk cache.


## 1. Anaconda environment
In Anaconda Prompt: `conda create -n arsenal-ml python=3.11 pandas numpy scikit-learn matplotlib requests ipykernel -y` then `conda activate arsenal-ml` and `python -m ipykernel install --user --name arsenal-ml`. Select this kernel in Jupyter. If packages are missing, run the next cell.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib requests ipywidgets joblib

In [ ]:
import os, json, time, getpass, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, classification_report
import joblib
warnings.filterwarnings('ignore', category=FutureWarning)
DATA_DIR = Path('arsenal_data'); DATA_DIR.mkdir(exist_ok=True)
print('Python / pandas:', __import__('sys').version.split()[0], pd.__version__)

## 2. Choose the competition and seasons

The dropdown is used by the download cell. **Only the selected competition is requested**, including for the optional Arsenal fixtures cell. The free tier may restrict older seasons. Set the earliest season to 2023 to avoid known 2022 access failures, or adjust it if your plan allows more history.

The target is **90-minute win/draw/loss**, not qualification after extra time or penalties. A Champions League-only model has less domestic context and may not contain Arsenal in every historical season.


In [ ]:
# Secure API entry; never save a secret in the notebook.
import ipywidgets as widgets
from IPython.display import display
API_KEY = os.getenv('FOOTBALL_DATA_API_KEY') or getpass.getpass('football-data.org API key (hidden): ')
BASE = 'https://api.football-data.org/v4'
HEADERS = {'X-Auth-Token': API_KEY.strip()}
COMPETITION_NAMES = {
    'CL':'Champions League', 'PL':'Premier League', 'ELC':'Championship',
    'BL1':'Bundesliga', 'PD':'La Liga', 'SA':'Serie A', 'FL1':'Ligue 1',
    'DED':'Eredivisie', 'PPL':'Primeira Liga', 'BSA':'Brazil Serie A',
    'EC':'European Championship', 'WC':'World Cup'
}
CURRENT_SEASON = datetime.now(timezone.utc).year if datetime.now(timezone.utc).month >= 7 else datetime.now(timezone.utc).year - 1
competition_picker = widgets.Dropdown(options=[(f'{name} ({code})',code) for code,name in COMPETITION_NAMES.items()],value='CL',description='Competition:',style={'description_width':'initial'},layout=widgets.Layout(width='380px'))
start_picker = widgets.IntSlider(value=min(2023,CURRENT_SEASON),min=2015,max=CURRENT_SEASON,description='Start season:',style={'description_width':'initial'},continuous_update=False,layout=widgets.Layout(width='480px'))
end_picker = widgets.IntSlider(value=CURRENT_SEASON,min=2015,max=CURRENT_SEASON,description='End season:',style={'description_width':'initial'},continuous_update=False,layout=widgets.Layout(width='480px'))
refresh_picker = widgets.Checkbox(value=True,description='Refresh latest season from API')
display(widgets.VBox([competition_picker,start_picker,end_picker,refresh_picker]))
DATA_DIR.mkdir(exist_ok=True)
REQUEST_PAUSE = 7
_last_request = [0.0]
# Respect API rate limits; do not cache failures.
_last_request = [0.0]
def api_get(path, params=None):
    for attempt in range(3):
        elapsed = time.monotonic() - _last_request[0]
        if elapsed < REQUEST_PAUSE:
            time.sleep(REQUEST_PAUSE-elapsed)
        try:
            response = requests.get(BASE+path, headers=HEADERS, params=params, timeout=30)
        except requests.RequestException as exc:
            if attempt == 2: raise RuntimeError(f'Network failure: {exc}') from exc
            time.sleep(5*(attempt+1)); continue
        _last_request[0] = time.monotonic()
        if response.status_code == 429:
            try: wait = max(65, int(response.headers.get('Retry-After','65')))
            except ValueError: wait = 65
            print('Rate limited; waiting',wait,'seconds')
            time.sleep(wait); continue
        if response.status_code != 200:
            raise RuntimeError(f'HTTP {response.status_code}: {response.text[:180]}')
        return response.json()
    raise RuntimeError('API rate limit persists after retries')


print('Select your competition above, then run the next download cell.')


## 3. Download the selected competition only

After changing the dropdown or sliders, **rerun this cell and all subsequent cells** to retrain. Requests for previous seasons are cached. If an older season returns 403, the downloader stops trying even older seasons; newer accessible seasons remain available.


In [ ]:
def get_season_matches(competition, season, refresh=False):
    file = DATA_DIR / f'{competition}_{season}.json'
    if file.exists() and not refresh:
        return json.loads(file.read_text())
    try:
        data = api_get(f'/competitions/{competition}/matches', {'season':season})
        matches = data.get('matches', [])
        if not isinstance(matches,list):
            raise RuntimeError('Unexpected API response')
        file.write_text(json.dumps(matches))
        return matches
    except Exception:
        if file.exists():
            print('  Using saved cache:', competition, season)
            return json.loads(file.read_text())
        raise

SELECTED_COMPETITION = competition_picker.value
START_SEASON = start_picker.value
END_SEASON = end_picker.value
if START_SEASON > END_SEASON:
    raise ValueError('Start season must not be after end season.')
print('Downloading ONLY:', COMPETITION_NAMES[SELECTED_COMPETITION], SELECTED_COMPETITION)
raw_matches = []
failures = []
public_frames = []  # Do not silently download other leagues as a fallback.
for season in range(END_SEASON, START_SEASON-1, -1):
    try:
        matches = get_season_matches(SELECTED_COMPETITION,season,
            refresh=refresh_picker.value and season==CURRENT_SEASON)
        raw_matches.extend(matches)
        print(f'{SELECTED_COMPETITION} {season}: {len(matches)} matches')
    except Exception as exc:
        failures.append((SELECTED_COMPETITION,season,str(exc)[:180]))
        print(f'Skipped {SELECTED_COMPETITION} {season}: {str(exc)[:130]}')
        if 'HTTP 403' in str(exc) and season != CURRENT_SEASON:
            print('Older seasons restricted; stopping historical requests.')
            break
        if 'HTTP 401' in str(exc):
            raise RuntimeError('Invalid API key; regenerate it and rerun configuration.') from exc
print('Downloaded match records:',len(raw_matches),'| Errors:',len(failures))
if not raw_matches:
    raise RuntimeError('No data for the selected competition. Try another season or competition and inspect the API error above.')


## 4. Clean and standardise matches
We identify teams by API IDs, not inconsistent names. Only finished games with full-time scores are used for training. For cup fixtures, the API full-time score can include extra time depending on the provider; **we exclude knockout matches with extra time** to preserve the 90-minute prediction target.

In [ ]:
records = []
for m in raw_matches:
    if m.get('status') != 'FINISHED':
        continue
    score = m.get('score') or {}
    full = score.get('fullTime') or {}
    regular = score.get('regularTime') or {}
    # Cup fixtures that required extra time are excluded, even when 90-min result is available.
    if score.get('duration') in ('EXTRA_TIME', 'PENALTY_SHOOTOUT'):
        continue
    if full.get('home') is None or full.get('away') is None:
        continue
    home, away = m.get('homeTeam') or {}, m.get('awayTeam') or {}
    if not home.get('id') or not away.get('id'):
        continue
    records.append({'match_id':m['id'], 'date':pd.to_datetime(m['utcDate'], utc=True),
        'competition':(m.get('competition') or {}).get('code', 'UNKNOWN'),
        'home_id':home['id'], 'away_id':away['id'],
        'home':home.get('name',''), 'away':away.get('name',''),
        'hg':int(full['home']), 'ag':int(full['away'])})
# Normalize public CSV records to the same match schema.
# Stable team IDs are generated from names for the CSV-only fallback.
if public_frames:
    csv_all=pd.concat(public_frames,ignore_index=True)
    csv_all['parsed_date']=pd.to_datetime(csv_all['Date'],dayfirst=True,errors='coerce')
    csv_all=csv_all.dropna(subset=['parsed_date','HomeTeam','AwayTeam','FTHG','FTAG'])
    all_names=sorted(set(csv_all['HomeTeam']).union(set(csv_all['AwayTeam'])))
    csv_ids={name:i+1000000 for i,name in enumerate(all_names)}
    for r in csv_all.itertuples(index=False):
        try:
            hg=int(r.FTHG); ag=int(r.FTAG)
            date=pd.Timestamp(r.parsed_date).tz_localize('UTC')
        except (ValueError,TypeError): continue
        records.append({'match_id':f'csv_{r.competition}_{r.season}_{r.Index}' if hasattr(r,'Index') else
                        f'csv_{r.competition}_{r.season}_{len(records)}',
                        'date':date,'competition':r.competition,
                        'home_id':csv_ids[r.HomeTeam],'away_id':csv_ids[r.AwayTeam],
                        'home':r.HomeTeam,'away':r.AwayTeam,'hg':hg,'ag':ag})
df = pd.DataFrame(records).drop_duplicates('match_id').sort_values(['date','match_id']).reset_index(drop=True)
if df.empty: raise RuntimeError('No finished matches found.')
print('Finished matches:', len(df), 'Range:', df.date.min(), 'to', df.date.max())
print('Competitions:', df.competition.value_counts().to_dict())
display(df.tail(10))

## 5. Leakage-free pre-match features
Each row uses only matches played **before that fixture date**. Teams in a given kickoff-date batch share the same pre-day ratings, preventing information from other same-day matches leaking into a prediction. Features include rolling form, goals for/against, Elo, number of prior matches, rest days and venue. Teams with no history are flagged as low confidence.

In [ ]:
from collections import defaultdict, deque
FEATURES = ['elo_diff','home_elo','away_elo','home_form','away_form','form_diff',
            'home_gf','away_gf','home_ga','away_ga','home_played','away_played',
            'home_rest','away_rest','home_advantage']

def initial_state():
    return {'elo':defaultdict(lambda:1500.0), 'form':defaultdict(lambda:deque(maxlen=5)),
            'gf':defaultdict(lambda:deque(maxlen=5)), 'ga':defaultdict(lambda:deque(maxlen=5)),
            'played':defaultdict(int), 'last':{}}

def team_features(state, tid, at):
    f = state['form'][tid]; gf = state['gf'][tid]; ga = state['ga'][tid]
    last = state['last'].get(tid)
    rest = min(max((at-last).days,0),30) if last is not None else 14
    return [state['elo'][tid], np.mean(f) if f else 1.0,
            np.mean(gf) if gf else 1.3, np.mean(ga) if ga else 1.3,
            state['played'][tid], rest]

def make_features(state, home_id, away_id, at):
    h=team_features(state,home_id,at); a=team_features(state,away_id,at)
    return [h[0]-a[0],h[0],a[0],h[1],a[1],h[1]-a[1],h[2],a[2],h[3],a[3],
            h[4],a[4],h[5],a[5],1]

def update_state(state, r):
    h,a = int(r.home_id),int(r.away_id); hg,ag=int(r.hg),int(r.ag)
    he,ae=state['elo'][h],state['elo'][a]
    expected=1/(1+10**((ae-he-65)/400))
    actual=1 if hg>ag else (0.5 if hg==ag else 0)
    delta=24*(actual-expected)
    state['elo'][h]+=delta; state['elo'][a]-=delta
    hp,ap=(3,0) if hg>ag else ((0,3) if hg<ag else (1,1))
    for tid,points,scored,conceded in [(h,hp,hg,ag),(a,ap,ag,hg)]:
        state['form'][tid].append(points)
        state['gf'][tid].append(scored)
        state['ga'][tid].append(conceded)
        state['played'][tid]+=1
        state['last'][tid]=r.date

def build_training_data(matches):
    state=initial_state(); X=[]; y=[]; meta=[]
    for day, group in matches.groupby(matches.date.dt.floor('D'),sort=True):
        # Calculate all pre-match features before updating any games on this date.
        for r in group.itertuples():
            X.append(make_features(state,int(r.home_id),int(r.away_id),r.date))
            y.append('H' if r.hg>r.ag else ('A' if r.hg<r.ag else 'D'))
            meta.append({'date':r.date,'match_id':r.match_id,'competition':r.competition,
                         'home':r.home,'away':r.away})
        for r in group.itertuples(): update_state(state,r)
    return pd.DataFrame(X,columns=FEATURES),np.array(y),pd.DataFrame(meta),state

X,y,meta,current_state=build_training_data(df)
print('Feature matrix:',X.shape)
display(X.tail())

## 6. Chronological train/test split and model selection
We reserve the most recent 20% of matches for testing, instead of shuffling historical and future games together. The model trains on *all teams*, not only Arsenal, so it can learn from opponents playing in their own leagues. We compare a simple multinomial logistic regression to a random forest and choose by **test log loss**. This test is exploratory: for a fully unbiased estimate, add a separate final holdout or rolling-origin validation.

In [ ]:
cut=int(len(X)*0.8)
if cut<100 or len(X)-cut<30: raise RuntimeError('Too few matches for reliable testing; obtain more accessible historical seasons.')
X_train,X_test=X.iloc[:cut],X.iloc[cut:]
y_train,y_test=y[:cut],y[cut:]
models={
    'Logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500)),
    'Random forest':RandomForestClassifier(n_estimators=250,min_samples_leaf=8,max_features=0.8,random_state=42,n_jobs=-1)
}
results=[]
for name, model in models.items():
    model.fit(X_train,y_train)
    probs=model.predict_proba(X_test)
    labels=model.classes_
    results.append({'model':name,'accuracy':accuracy_score(y_test,model.predict(X_test)),
                    'log_loss':log_loss(y_test,probs,labels=labels)})
metrics=pd.DataFrame(results).sort_values('log_loss')
display(metrics)
best_name=metrics.iloc[0]['model']; best_model=models[best_name]
print('Selected model:',best_name)
print(classification_report(y_test,best_model.predict(X_test),zero_division=0))

## 7. Fit the final model on all available finished matches
The test scores above are historical; the final model below is refitted on all completed games so that new results inform predictions. Re-run the download, feature engineering and fitting cells after games finish.

In [ ]:
final_model=models[best_name]
final_model.fit(X,y)
joblib.dump({'model':final_model,'features':FEATURES,'trained_at':datetime.now(timezone.utc).isoformat()},DATA_DIR/'arsenal_model.joblib')
team_names={}
for r in df.itertuples():
    team_names[int(r.home_id)]=r.home; team_names[int(r.away_id)]=r.away
arsenal_ids=[tid for tid,name in team_names.items() if 'arsenal' in name.lower()]
if not arsenal_ids: raise RuntimeError('Arsenal not present in accessible data.')
ARSENAL_ID=arsenal_ids[0]
print('Arsenal team ID:',ARSENAL_ID,'| Known teams:',len(team_names))

## 8. Manual Arsenal prediction (any team in the downloaded data)
For clubs with limited historical matches, output a warning. Home/away refers to **Arsenal**. Neutral venues use a simple averaging of both home orientations; this is an approximation, not a venue-trained model.

In [ ]:
def find_team(name):
    matches=[(tid,label) for tid,label in team_names.items() if name.lower() in label.lower()]
    if len(matches)==1: return matches[0]
    if not matches: raise ValueError(f'No team matching {name!r}. Download its league first.')
    raise ValueError('Ambiguous team: '+str(matches[:20]))

def predict_arsenal(opponent, venue='home', show_plot=True):
    opp_id,opp_name=find_team(opponent)
    if opp_id==ARSENAL_ID: raise ValueError('Choose an opponent other than Arsenal.')
    at=pd.Timestamp.now(tz='UTC')
    def calc(home_id,away_id):
        row=pd.DataFrame([make_features(current_state,home_id,away_id,at)],columns=FEATURES)
        probabilities=dict(zip(final_model.classes_,final_model.predict_proba(row)[0]))
        return probabilities
    if venue=='home':
        p=calc(ARSENAL_ID,opp_id); result=[p['H'],p['D'],p['A']]
    elif venue=='away':
        p=calc(opp_id,ARSENAL_ID); result=[p['A'],p['D'],p['H']]
    elif venue=='neutral':
        p1=calc(ARSENAL_ID,opp_id); p2=calc(opp_id,ARSENAL_ID)
        result=[(p1['H']+p2['A'])/2,(p1['D']+p2['D'])/2,(p1['A']+p2['H'])/2]
    else: raise ValueError('venue must be home, away, or neutral')
    result=pd.Series(result,index=['Arsenal win','Draw',f'{opp_name} win'])
    print(f'Arsenal vs {opp_name} ({venue})')
    print((result*100).round(1).astype(str)+'%')
    if current_state['played'][opp_id]<8:
        print('WARNING: Limited opponent history; prediction is less reliable.')
    if show_plot:
        ax=result.mul(100).plot(kind='barh',figsize=(8,3),xlim=(0,100),title='Predicted 90-minute result probabilities')
        ax.set_xlabel('Probability (%)'); plt.tight_layout();plt.show()
    return result

# Example: uncomment after training
# predict_arsenal('Liverpool', 'home')

## 9. Upcoming Arsenal fixtures (as provided by your API plan)
Retrieve future fixtures across accessible competitions, then choose one by its index. Some competitions may not provide fixtures or may require paid access. This makes predictions for upcoming games, not live in-play odds.

In [ ]:
# Request fixtures ONLY for the competition selected above.
upcoming=[]
try:
    data=api_get(f'/competitions/{SELECTED_COMPETITION}/matches',{'status':'SCHEDULED'})
    for m in data.get('matches',[]):
        h=m.get('homeTeam') or {}; a=m.get('awayTeam') or {}
        if ARSENAL_ID in [h.get('id'),a.get('id')]:
            upcoming.append({'date':m.get('utcDate'),'competition':SELECTED_COMPETITION,
                             'home_id':h.get('id'),'away_id':a.get('id'),
                             'home':h.get('name'),'away':a.get('name')})
except Exception as exc:
    print('Fixture access unavailable:',str(exc)[:130])
fixtures=pd.DataFrame(upcoming).drop_duplicates() if upcoming else pd.DataFrame()
if len(fixtures):
    fixtures=fixtures.sort_values('date').reset_index(drop=True)
    display(fixtures[['date','competition','home','away']])
else:
    print('No upcoming Arsenal fixtures in',SELECTED_COMPETITION)


In [ ]:
def predict_fixture(index):
    if fixtures.empty: raise ValueError('No fixtures available')
    f=fixtures.iloc[index]
    opponent_id=f.away_id if f.home_id==ARSENAL_ID else f.home_id
    if opponent_id not in team_names:
        raise ValueError('Opponent has no historical match data; download its league or cup first.')
    venue='home' if f.home_id==ARSENAL_ID else 'away'
    print('Kickoff:',f.date,'| Competition:',f.competition)
    return predict_arsenal(team_names[opponent_id],venue)
# Example: predict_fixture(0)

## 10. Interactive controls (optional)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
opponents=sorted([name for tid,name in team_names.items() if tid!=ARSENAL_ID])
team_widget=widgets.Combobox(options=opponents,placeholder='Type opponent name',description='Opponent:')
venue_widget=widgets.Dropdown(options=['home','away','neutral'],description='Venue:')
button=widgets.Button(description='Predict Arsenal',button_style='success')
output=widgets.Output()
def on_predict(_):
    with output:
        output.clear_output(wait=True)
        try: predict_arsenal(team_widget.value,venue_widget.value)
        except Exception as exc: print('Error:',exc)
button.on_click(on_predict)
display(widgets.VBox([team_widget,venue_widget,button,output]))

## 11. Refreshing and interpreting the model

Change the competition or season range in the selector cell, then rerun the download cell **and every subsequent cell**. Only that competition is downloaded; previous successful downloads are reused from the local `arsenal_data` cache. Refreshing the current season fetches new results when you rerun the notebook.

**Limitations:** The model is trained on the selected competition only, so Champions League-only ratings and form omit domestic league performances. If Arsenal is absent in the selected dataset, Arsenal predictions cannot be made; choose a competition containing Arsenal. Older seasons may be restricted by your subscription. The model estimates 90-minute results, not cup progression. This is not a live background updater.

[football-data.org documentation](https://www.football-data.org/documentation/api)
